In [9]:
from sklearn.impute import KNNImputer
from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd

In [17]:
df = pd.read_csv("synthetic_dataset.csv")
df_num = df.select_dtypes(include = np.number) #int and float
complete_rows = df_num.dropna()
complete_rows

,age,purchase_amount_kes,quantity,satisfaction_score
0,29.0,6836.99,7.0,1.0
1,56.0,4539.52,8.0,3.0
2,19.0,5064.24,1.0,1.0
4,54.0,11335.14,1.0,3.0
6,19.0,11886.35,1.0,3.0
8,45.0,7517.42,9.0,4.0
10,54.0,8333.03,1.0,2.0
11,49.0,6734.65,1.0,4.0
12,50.0,13338.02,2.0,3.0
16,61.0,11470.36,5.0,1.0


In [23]:
#choosing what values to hide
np.random.seed(1) #seed is same random mask every time you run it
mask = np.random.rand(*complete_rows.shape) < 0.10

In [24]:
#make a copy and an artificial missing value
df_test = complete_rows.copy()
df_test[mask] = np.nan
df_test[mask]

,age,purchase_amount_kes,quantity,satisfaction_score
0,29.0,6836.99,NaN,1.0
1,56.0,NaN,8.0,3.0
4,54.0,11335.14,NaN,3.0
10,54.0,8333.03,NaN,NaN
10,54.0,8333.03,NaN,NaN
11,49.0,6734.65,NaN,4.0
16,61.0,11470.36,NaN,1.0
19,23.0,1444.78,NaN,2.0
20,49.0,10574.34,5.0,NaN
26,NaN,14802.27,5.0,1.0


In [40]:
#running knn on fake imputed data
imputer = KNNImputer(n_neighbors = 5)
df_test_imputed = pd.DataFrame(imputer.fit_transform(df_test), columns = complete_rows.columns)
df_test_imputed

,age,purchase_amount_kes,quantity,satisfaction_score
0,29.0,6836.990,6.8,1.0
1,56.0,10843.868,8.0,3.0
2,19.0,5064.240,1.0,1.0
3,54.0,11335.140,4.8,3.0
4,19.0,11886.350,1.0,3.0
5,45.0,7517.420,9.0,4.0
6,54.0,8333.030,4.8,2.8
7,49.0,6734.650,6.8,4.0
8,50.0,13338.020,2.0,3.0
9,61.0,11470.360,4.6,1.0


In [41]:
#pulling just the masked values
true_vals = complete_rows.values[mask] #true values
imputed_vals =df_test_imputed.values[mask] #what KNN guessed on the artificial blanks

In [42]:
#comparison btwn true and guessed values
mae = mean_absolute_error(true_vals, imputed_vals)
print("MAE:", mae)

MAE: 653.4293793103448


In [43]:
# Per-column MAE (so one big-scale column doesn't dominate the result)
for col in complete_rows.columns:
    col_idx = complete_rows.columns.get_loc(col)
    col_mask = mask[:, col_idx]
    if col_mask.sum() > 0:
        col_mae = mean_absolute_error(
            complete_rows[col].values[col_mask],
            df_test_imputed[col].values[col_mask]
        )
        print(f"{col}: MAE = {col_mae:.2f}")

# Compare to a dumb baseline (just filling with the column mean)
baseline_vals = np.take(complete_rows.mean().values, np.where(mask)[1])
baseline_mae = mean_absolute_error(true_vals, baseline_vals)
print("Baseline (mean-fill) MAE:", baseline_mae)
print("Is KNN better than just using the mean?", mae < baseline_mae)

age: MAE = 20.80
purchase_amount_kes: MAE = 3759.41
quantity: MAE = 3.09
satisfaction_score: MAE = 1.04
Baseline (mean-fill) MAE: 774.5462613430124
Is KNN better than just using the mean? True


In [50]:
#for categorical data
from sklearn.impute import SimpleImputer
df = pd.read_csv("synthetic_dataset.csv")
#columns to mode impute
mode_cols = ['city', 'product_category', 'payment_method', 'is_repeat_customer'] 
#columns to fill with unknown
unknown_cols = ['first_name', 'last_name']
#mode imputation
cat_imp = SimpleImputer(strategy = 'most_frequent')
df_mode_filled = pd.DataFrame(cat_imp.fit_transform(df[mode_cols]), columns = mode_cols)

In [52]:
#filling names with unknown
df_names_filled = df[unknown_cols].fillna("Unknown")
df_names_filled

,first_name,last_name
0,Michael,Otieno
1,Unknown,Chebet
2,Njoroge,Wambui
3,David,Unknown
4,Linda,Odhiambo
...,...,...
95,Susan,Wambui
96,Mary,Kariuki
97,James,Wanjiru
98,Achieng,Mwangi


In [56]:
#combining the dataframe
df_cat_filled = pd.concat([df_names_filled.reset_index(drop = True), df_mode_filled.reset_index(drop = True)], axis = 1)
df_cat_filled

,first_name,last_name,city,product_category,payment_method,is_repeat_customer
0,Michael,Otieno,Nairobi,Fashion,Card,True
1,Unknown,Chebet,Nakuru,Electronics,Card,True
2,Njoroge,Wambui,Nairobi,Electronics,Bank Transfer,False
3,David,Unknown,Eldoret,Toys,Bank Transfer,False
4,Linda,Odhiambo,Nakuru,Groceries,Card,False
...,...,...,...,...,...,...
95,Susan,Wambui,Nakuru,Home & Living,Card,True
96,Mary,Kariuki,Nairobi,Fashion,Cash,False
97,James,Wanjiru,Nyeri,Groceries,Card,False
98,Achieng,Mwangi,Machakos,Books,Bank Transfer,False


In [57]:
#sanity check
print(df_cat_filled.isna().sum())

first_name            0
last_name             0
city                  0
product_category      0
payment_method        0
is_repeat_customer    0
dtype: int64
